# 08.0 Failure analysis

## Notebook overview
This notebook loads the saved main comparison and threshold outputs, identifies the weakest category for the best overall method, rebuilds only that runtime, and exports a compact qualitative failure review.

## Purpose
- load the shared split manifest and leakage checks from notebook 01
- load the saved main comparison outputs from notebook 05
- load the balanced threshold outputs from notebook 06
- identify the weakest category for the strongest overall method
- save a detailed failure table, a compact error summary, and a qualitative failure figure

## Process
1. resolve the dataset path and load the prior notebook artefacts.
2. rebuild only the required runtime for the selected method and category.
3. score the full test set and assign error types using the saved balanced threshold.
4. export a detailed failure table, a summary table, and a small figure of representative examples.


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Optional dependency install
#------------------------------------------------------------------------------
try:
    import faiss
    print("faiss already available")
except Exception:
    !pip -q install -U faiss-cpu


In [ ]:
#------------------------------------------------------------------------------
# 1.1 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import random
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

import faiss
from IPython.display import display


# Print a clean banner so long notebook output is easier to scan.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create the parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read a JSON file from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return file size in MB if the path exists.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Keep deterministic behaviour consistent with the earlier notebooks.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Build a stable category-specific seed from text.
def stable_seed_from_text(seed, text):
    key = f"{seed}_{text}".encode("utf-8")
    return int(hashlib.md5(key).hexdigest()[:8], 16)


# Reset tracked peak CUDA memory before one fit stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Read the tracked peak CUDA memory in MB.
def get_peak_gpu_memory_mb():
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan


print_banner("Environment")
print("Python        :", sys.version.split()[0])
print("Torch         :", torch.__version__)
print("Torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count     :", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    print(f"GPU {gpu_idx} name   :", torch.cuda.get_device_name(gpu_idx))


# 2.0 Run settings

## Purpose
- define the failure-analysis settings in one place before any runtime rebuilding starts
- keep the notebook portable across Kaggle and local use
- save outputs into a clean folder structure that matches the earlier hardened notebooks


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "08_failure_analysis"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16

BACKBONE_NAME = "resnet18"
FEATURE_LAYER_NAME = "layer3"
CORESET_RATIO = 0.05
MAX_TRAIN_PATCHES = 120_000
PATCH_SCORE_TOPK = 200
PADIM_DIM = 100
PADIM_EPS = 1e-4

FAILURE_POLICY = "balanced"
TOP_FP_N = 2
TOP_FN_N = 2
TOP_TP_N = 1

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
NOTEBOOK_03_ROOT = WORK_ROOT / "03_autoencoder_baseline" / RUN_MODE
NOTEBOOK_04_ROOT = WORK_ROOT / "04_simclr_pretraining" / RUN_MODE
NOTEBOOK_05_ROOT = WORK_ROOT / "05_ssl_downstream_models" / RUN_MODE
NOTEBOOK_06_ROOT = WORK_ROOT / "06_main_comparison_and_thresholds" / RUN_MODE

SPLIT_MANIFEST_PATH = NOTEBOOK_01_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = NOTEBOOK_01_ROOT / "json" / "leakage_report.json"

AE_CHECKPOINTS_DIR = NOTEBOOK_03_ROOT / "checkpoints"
SSL_CHECKPOINTS_DIR = NOTEBOOK_04_ROOT / "checkpoints"

TABLE_MAIN_COMPARISON_MEAN_DEP_PATH = NOTEBOOK_05_ROOT / "tables" / "table_main_comparison_mean.csv"
TABLE_MAIN_COMPARISON_FULL_DEP_PATH = NOTEBOOK_05_ROOT / "tables" / "table_main_comparison_full.csv"
TABLE_THRESHOLDS_CATEGORY_DEP_PATH = NOTEBOOK_06_ROOT / "tables" / "table_thresholds_category.csv"
TABLE_THRESHOLDS_MEAN_DEP_PATH = NOTEBOOK_06_ROOT / "tables" / "table_thresholds_mean.csv"
THRESHOLD_TARGETS_DEP_PATH = NOTEBOOK_06_ROOT / "json" / "threshold_targets.json"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
FAILURE_SELECTION_PATH = JSON_DIR / "failure_selection.json"
AVAILABLE_SSL_CHECKPOINTS_PATH = JSON_DIR / "available_ssl_checkpoints.json"
TABLE_FAILURES_PATH = TABLES_DIR / "table_failure_examples.csv"
TABLE_FAILURE_COUNTS_PATH = TABLES_DIR / "table_failure_counts.csv"
FIG_FAILURES_PATH = FIGURES_DIR / "fig_failure_examples.png"


# 3.0 Dataset and prior artefacts

## Purpose
- resolve the dataset path in a way that works in Kaggle or a local clone
- load the split manifest and leakage report from notebook 01
- load the saved main comparison outputs from notebook 05
- load the threshold outputs from notebook 06 before any runtime rebuilding starts


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve dataset path and load prior notebook artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

required_paths = [
    SPLIT_MANIFEST_PATH,
    LEAKAGE_REPORT_PATH,
    TABLE_MAIN_COMPARISON_MEAN_DEP_PATH,
    TABLE_MAIN_COMPARISON_FULL_DEP_PATH,
    TABLE_THRESHOLDS_CATEGORY_DEP_PATH,
]
missing_required = [str(path) for path in required_paths if not Path(path).exists()]
if missing_required:
    raise FileNotFoundError(
        "One or more dependency artefacts are missing. Run notebooks 01, 05, and 06 first.\n"
        + "\n".join(missing_required)
    )

split_manifest = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)
table_main_comparison_mean = pd.read_csv(TABLE_MAIN_COMPARISON_MEAN_DEP_PATH)
table_main_comparison_full = pd.read_csv(TABLE_MAIN_COMPARISON_FULL_DEP_PATH)
table_thresholds_category = pd.read_csv(TABLE_THRESHOLDS_CATEGORY_DEP_PATH)

table_thresholds_mean = None
if TABLE_THRESHOLDS_MEAN_DEP_PATH.exists():
    table_thresholds_mean = pd.read_csv(TABLE_THRESHOLDS_MEAN_DEP_PATH)

threshold_targets = None
if THRESHOLD_TARGETS_DEP_PATH.exists():
    threshold_targets = read_json(THRESHOLD_TARGETS_DEP_PATH)

leakage_failures = []
for key, value in leakage_report.items():
    if isinstance(value, (int, float)) and value != 0:
        leakage_failures.append((key, value))
    elif isinstance(value, list) and len(value) > 0:
        leakage_failures.append((key, f"{len(value)} records"))

if leakage_failures:
    raise RuntimeError(
        "Notebook 01 reported non-zero leakage checks. Fix notebook 01 before continuing.\n"
        + "\n".join([f"{k}: {v}" for k, v in leakage_failures])
    )

ssl_ckpt_map = {
    "mild": SSL_CHECKPOINTS_DIR / "simclr_mild_encoder.pt",
    "strong": SSL_CHECKPOINTS_DIR / "simclr_strong_encoder.pt",
}
available_ssl_ckpt_map = {tag: str(path) for tag, path in ssl_ckpt_map.items() if path.exists()}
save_json_overwrite(available_ssl_ckpt_map, AVAILABLE_SSL_CHECKPOINTS_PATH)

categories_from_manifest = list(split_manifest.keys())
CATEGORIES_ACTIVE = [cat for cat in CATEGORIES_ACTIVE if cat in categories_from_manifest]
if len(CATEGORIES_ACTIVE) == 0:
    raise RuntimeError("No active categories from the run configuration were found in the split manifest.")

print_banner("Loaded prior artefacts")
print("Dataset root                  :", MVTEC_DIR)
print("Split manifest path           :", SPLIT_MANIFEST_PATH)
print("Leakage report path           :", LEAKAGE_REPORT_PATH)
print("Main comparison mean path     :", TABLE_MAIN_COMPARISON_MEAN_DEP_PATH)
print("Main comparison full path     :", TABLE_MAIN_COMPARISON_FULL_DEP_PATH)
print("Threshold category path       :", TABLE_THRESHOLDS_CATEGORY_DEP_PATH)
print("Threshold targets path        :", THRESHOLD_TARGETS_DEP_PATH if THRESHOLD_TARGETS_DEP_PATH.exists() else "not found")
print("Available SSL checkpoints     :", list(available_ssl_ckpt_map.keys()))
print("Active categories             :", CATEGORIES_ACTIVE)
print("Main comparison mean rows     :", len(table_main_comparison_mean))
print("Main comparison full rows     :", len(table_main_comparison_full))
print("Threshold category rows       :", len(table_thresholds_category))
display(table_main_comparison_mean.head())


# 4.0 Shared datasets, transforms, metrics, and visualisation helpers

## Purpose
- define the shared dataset wrapper and dataloaders used to rebuild only the selected failure-analysis runtime
- keep image preprocessing aligned with the earlier notebooks
- add small visualisation helpers for the exported qualitative failure figure


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Shared transforms, dataset class, loaders, metrics, and plotting helpers
#------------------------------------------------------------------------------
tfm_imagenet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tfm_ae = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


# Load one RGB image.
def load_rgb(path):
    return Image.open(path).convert("RGB")


# Load one binary mask or return an all-zero mask for good images.
def load_mask(path):
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

    mask = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(mask)[None, :, :]


# Return items in a consistent format across the shared split manifest.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if self.mode in ["train_good", "val_good"]:
            img_path = item
            label = 0
            mask_path = None
        else:
            img_path = item["img_path"]
            label = int(item["label"])
            mask_path = item.get("mask_path", None)

        image = self.img_tfm(load_rgb(img_path))
        mask = load_mask(mask_path)
        return image, int(label), mask, str(img_path)


# Build a DataLoader matched to the active runtime.
def make_loader(items, mode, input_kind, batch_size, shuffle):
    img_tfm = tfm_imagenet if input_kind == "imagenet" else tfm_ae
    dataset = MvtecDataset(items=items, mode=mode, img_tfm=img_tfm)

    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE == "cuda",
    }

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**loader_kwargs)


# Build the train, validation, and test loaders for one category.
def make_split_loaders(category, input_kind):
    train_loader = make_loader(split_manifest[category]["train_good"], mode="train_good", input_kind=input_kind, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = make_loader(split_manifest[category]["val_good"], mode="val_good", input_kind=input_kind, batch_size=BATCH_SIZE_TEST, shuffle=False)
    test_loader = make_loader(split_manifest[category]["test"], mode="test", input_kind=input_kind, batch_size=BATCH_SIZE_TEST, shuffle=False)
    return train_loader, val_loader, test_loader


# Compute image-level ROC-AUC safely.
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Compute image-level PR-AUC safely.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


# Flatten masks and heatmaps to compute pixel ROC-AUC.
def pixel_roc_auc(masks_list, heatmaps_list):
    if len(masks_list) == 0 or len(heatmaps_list) == 0:
        return np.nan

    y_true = np.concatenate([mask.reshape(-1) for mask in masks_list]).astype(int)
    y_score = np.concatenate([heat.reshape(-1) for heat in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Evaluate one scoring function across the test set.
def eval_method(test_loader, score_fn):
    y_true = []
    y_score = []
    masks = []
    heats = []

    eval_start = time.time()
    eval_image_n = 0

    for images, labels, batch_masks, _ in test_loader:
        scores, heatmaps = score_fn(images)

        y_true.extend(labels.numpy().tolist())
        y_score.extend(scores.tolist())

        mask_np = batch_masks.squeeze(1).numpy()
        for idx in range(mask_np.shape[0]):
            masks.append(mask_np[idx])
            heats.append(heatmaps[idx])

        eval_image_n += images.shape[0]

    total_eval_sec = time.time() - eval_start

    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heats),
        "sec_per_img": total_eval_sec / max(eval_image_n, 1),
        "n_eval_images": int(eval_image_n),
        "y_true": y_true,
        "y_score": y_score,
        "masks": masks,
        "heats": heats,
    }


# Collect one image-level score per item for threshold analysis.
def collect_image_scores(loader, score_fn):
    rows = []
    for images, labels, _, paths in loader:
        scores, _ = score_fn(images)
        for idx in range(len(paths)):
            rows.append({
                "path": paths[idx],
                "label": int(labels[idx].item()),
                "score": float(scores[idx]),
            })
    return pd.DataFrame(rows)


# Compute thresholded metrics at one cut-off.
def threshold_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    normal_mask = (y_true == 0)
    fp = np.sum((y_pred == 1) & normal_mask)
    n_normal = np.sum(normal_mask)
    fpr = fp / max(n_normal, 1)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
    }


# Normalise an array to the 0-1 range for display.
def norm_01(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

# Undo tensor formatting so images display correctly.
def tensor_to_display(img_tensor):
    img = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    if img.min() < 0 or img.max() > 1.5:
        img = img * np.array(IMAGENET_STD)[None, None, :] + np.array(IMAGENET_MEAN)[None, None, :]
    return np.clip(img, 0.0, 1.0)

# Draw an image with an optional heatmap overlay.
def overlay(ax, img_tensor, heat_2d=None, title=""):
    ax.imshow(tensor_to_display(img_tensor))
    if heat_2d is not None:
        ax.imshow(norm_01(heat_2d), alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

# Show a mask with a clean grayscale colourmap.
def show_mask(ax, mask_2d, title=""):
    ax.imshow(mask_2d, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

# Collect detailed predictions so specific examples can be reviewed later.
def collect_detailed_preds(loader, score_fn):
    rows = []
    for images, labels, masks, paths in loader:
        scores, heats = score_fn(images)
        for idx in range(len(paths)):
            rows.append({
                "path": str(paths[idx]),
                "label": int(labels[idx].item()),
                "score": float(scores[idx]),
                "img_tensor": images[idx].detach().cpu(),
                "mask": masks[idx].squeeze(0).numpy(),
                "heat": heats[idx],
            })
    return rows


# 5.0 Runtime builders

## Purpose
- rebuild only the method runtime required for the selected failure analysis
- keep the method reconstruction logic aligned with notebook 06 so the failure examples come from the same pipeline assumptions


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Backbones, PatchCore, PaDiM, autoencoder, and runtime builders
#------------------------------------------------------------------------------
def get_resnet18_ssl():
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Identity()
    model.eval()
    return model


# Build the ImageNet backbone used in the baseline notebooks.
def get_resnet_imagenet():
    try:
        model = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        )
    except Exception as exc:
        raise RuntimeError(
            "ImageNet weights could not be loaded for resnet18. "
            "Run the notebook in an environment where torchvision pretrained weights are available."
        ) from exc

    model.fc = nn.Identity()
    model.eval()
    return model


# Register hooks so intermediate feature maps can be collected.
def make_feature_hooks(model, layer_names):
    features = {}
    handles = []

    def hook_factory(layer_name):
        def _hook(_, __, output):
            features[layer_name] = output
        return _hook

    for layer_name in layer_names:
        handle = getattr(model, layer_name).register_forward_hook(hook_factory(layer_name))
        handles.append(handle)

    return features, handles


# Remove hook handles after the runtime is no longer needed.
def remove_handles(handles):
    for handle in handles:
        handle.remove()


@torch.inference_mode()
def forward_get_feats(model, features, inputs, layer_names):
    _ = model(inputs)
    return [features[layer_name] for layer_name in layer_names]


# Flatten a feature map into patch rows.
def fmap_to_patches(feature_map):
    batch_n, channels, height, width = feature_map.shape
    return feature_map.permute(0, 2, 3, 1).reshape(batch_n, height * width, channels)


# Upsample and concatenate patch features from one or more layers.
def concat_patch_features(feature_maps):
    target_h, target_w = feature_maps[-1].shape[-2:]
    patch_list = []

    for feature_map in feature_maps:
        if feature_map.shape[-2:] != (target_h, target_w):
            feature_map = F.interpolate(
                feature_map,
                size=(target_h, target_w),
                mode="bilinear",
                align_corners=False,
            )
        patch_list.append(fmap_to_patches(feature_map))

    return torch.cat(patch_list, dim=-1)


# Build a FAISS L2 index for nearest-neighbour search.
def faiss_index_l2(array_2d):
    array_2d = np.asarray(array_2d, dtype=np.float32)
    index = faiss.IndexFlatL2(array_2d.shape[1])
    index.add(array_2d)
    return index


# Rebuild one saved SimCLR encoder checkpoint.
def load_ssl_encoder(ckpt_path):
    model = get_resnet18_ssl()
    state = torch.load(ckpt_path, map_location="cpu")

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if isinstance(state, dict):
        clean_state = {}
        for key, value in state.items():
            clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
            clean_state[clean_key] = value
        state = clean_state

    model.load_state_dict(state, strict=True)
    model.eval()
    return model


# Collect normal train patch embeddings and keep a deterministic coreset subset.
def build_patch_bank(model, features, train_loader, layer_names, category, source_tag):
    patch_chunks = []
    total_patches = 0

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_maps = forward_get_feats(model, features, images, layer_names)
        patches = concat_patch_features(feature_maps).detach().cpu().numpy()
        patches = patches.reshape(-1, patches.shape[-1]).astype(np.float32)
        patch_chunks.append(patches)
        total_patches += patches.shape[0]

        if total_patches >= MAX_TRAIN_PATCHES:
            break

    bank_full = np.concatenate(patch_chunks, axis=0).astype(np.float32)

    if len(bank_full) > MAX_TRAIN_PATCHES:
        rng_cap = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{source_tag}_patch_bank_cap"))
        idx_cap = rng_cap.choice(len(bank_full), size=MAX_TRAIN_PATCHES, replace=False)
        bank_full = bank_full[idx_cap]

    keep_n = max(1, int(round(len(bank_full) * float(CORESET_RATIO))))
    keep_n = min(keep_n, len(bank_full))

    rng_core = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{source_tag}_patchcore_coreset"))
    coreset_idx = rng_core.choice(len(bank_full), size=keep_n, replace=False)
    bank = bank_full[coreset_idx]

    return bank


@torch.inference_mode()
def patchcore_scores(model, features, images, layer_names, index):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_maps = forward_get_feats(model, features, images, layer_names)
    patches = concat_patch_features(feature_maps).detach().cpu().numpy()

    batch_n, patch_n, feat_dim = patches.shape
    queries = patches.reshape(-1, feat_dim).astype(np.float32)
    dist2, _ = index.search(queries, 1)
    dist2 = dist2.reshape(batch_n, patch_n)

    feat_h, feat_w = feature_maps[-1].shape[-2:]
    heat = dist2.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


# Fit PaDiM Gaussian statistics from normal train patches.
def padim_fit(model, features, train_loader, layer_name, category, source_tag):
    all_feature_maps = []

    for images, _, _, _ in train_loader:
        images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
        feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()
        all_feature_maps.append(feature_map)

    feature_map = torch.cat(all_feature_maps, dim=0)
    sample_n, channels, feat_h, feat_w = feature_map.shape

    dim = min(PADIM_DIM, channels)
    rng_dim = np.random.default_rng(stable_seed_from_text(SEED, f"{category}_{source_tag}_padim_dim"))
    dim_idx = rng_dim.choice(channels, size=dim, replace=False)

    feature_map = feature_map[:, dim_idx, :, :]
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(sample_n, feat_h * feat_w, dim).numpy()

    mu = feats_np.mean(axis=0)
    cov_inv = []

    eye = np.eye(dim, dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        patch_features = feats_np[:, patch_idx, :]
        cov = np.cov(patch_features, rowvar=False).astype(np.float32) + PADIM_EPS * eye
        cov_inv.append(np.linalg.inv(cov).astype(np.float32))

    cov_inv = np.stack(cov_inv, axis=0)

    return {
        "dim_idx": dim_idx.astype(np.int64),
        "mu": mu.astype(np.float32),
        "cov_inv": cov_inv.astype(np.float32),
        "H": int(feat_h),
        "W": int(feat_w),
        "D": int(dim),
    }


@torch.inference_mode()
def padim_scores(model, features, images, layer_name, stats):
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))
    feature_map = forward_get_feats(model, features, images, [layer_name])[0].detach().cpu()

    dim_idx = torch.tensor(stats["dim_idx"])
    feature_map = feature_map[:, dim_idx, :, :]

    batch_n, feat_dim, feat_h, feat_w = feature_map.shape
    feats_np = feature_map.permute(0, 2, 3, 1).reshape(batch_n, feat_h * feat_w, feat_dim).numpy()

    mu = stats["mu"]
    cov_inv = stats["cov_inv"]

    heat = np.zeros((batch_n, feat_h * feat_w), dtype=np.float32)
    for patch_idx in range(feat_h * feat_w):
        diff = feats_np[:, patch_idx, :] - mu[patch_idx]
        tmp = diff @ cov_inv[patch_idx]
        heat[:, patch_idx] = np.sum(tmp * diff, axis=1)

    heat = heat.reshape(batch_n, feat_h, feat_w)
    heat_tensor = torch.from_numpy(heat).unsqueeze(1)
    heat_up = F.interpolate(
        heat_tensor,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1).numpy()

    flat = heat_up.reshape(batch_n, -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heat_up


class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.dec(self.enc(x))


@torch.inference_mode()
def ae_scores(model, images):
    model.eval()
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))

    recon = model(images)
    err = (recon - images).pow(2).mean(dim=1)

    heatmaps = err.detach().cpu().numpy()
    flat = heatmaps.reshape(heatmaps.shape[0], -1)
    topk_n = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk_n:], axis=1)

    return scores, heatmaps


# Translate the method label back into the settings needed to rebuild it.
def parse_method_name(method_name):
    if method_name == "imagenet_patchcore":
        return {"representation_source": "imagenet", "anomaly_head": "patchcore", "aug_strength": "none"}
    if method_name == "imagenet_padim":
        return {"representation_source": "imagenet", "anomaly_head": "padim", "aug_strength": "none"}
    if method_name == "autoencoder":
        return {"representation_source": "none", "anomaly_head": "autoencoder", "aug_strength": "none"}

    _, aug_strength, anomaly_head = method_name.split("_")
    return {"representation_source": "simclr", "anomaly_head": anomaly_head, "aug_strength": aug_strength}


# Build the loaders, backbone, and score function for one feature-based method.
def build_feature_method_runtime(category, representation_source, anomaly_head, aug_strength=None):
    train_loader, val_loader, test_loader = make_split_loaders(category, input_kind="imagenet")

    if representation_source == "imagenet":
        model = get_resnet_imagenet()
        ckpt_path = None
        source_tag = "imagenet"
    else:
        ckpt_path = Path(available_ssl_ckpt_map[aug_strength])
        model = load_ssl_encoder(ckpt_path)
        source_tag = aug_strength

    model = model.to(DEVICE)
    features, handles = make_feature_hooks(model, [FEATURE_LAYER_NAME])

    reset_peak_gpu_memory()
    fit_start = time.time()

    if anomaly_head == "patchcore":
        bank = build_patch_bank(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_names=[FEATURE_LAYER_NAME],
            category=category,
            source_tag=source_tag,
        )
        index = faiss_index_l2(bank)

        def score_fn(images):
            return patchcore_scores(model, features, images, [FEATURE_LAYER_NAME], index)

        fit_stats = {
            "feature_dim": int(bank.shape[1]),
            "memory_bank_n": int(bank.shape[0]),
            "memory_bank_mb": float(bank.nbytes / (1024 ** 2)),
        }

    elif anomaly_head == "padim":
        stats = padim_fit(
            model=model,
            features=features,
            train_loader=train_loader,
            layer_name=FEATURE_LAYER_NAME,
            category=category,
            source_tag=source_tag,
        )

        def score_fn(images):
            return padim_scores(model, features, images, FEATURE_LAYER_NAME, stats)

        fit_stats = {
            "feature_dim": int(stats["D"]),
            "memory_bank_n": int(stats["H"] * stats["W"]),
            "memory_bank_mb": float((stats["mu"].nbytes + stats["cov_inv"].nbytes) / (1024 ** 2)),
        }

    else:
        remove_handles(handles)
        raise ValueError(f"Unsupported anomaly head: {anomaly_head}")

    fit_sec = float(time.time() - fit_start)
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "features": features,
        "handles": handles,
        "score_fn": score_fn,
        "feature_dim": fit_stats["feature_dim"],
        "memory_bank_n": fit_stats["memory_bank_n"],
        "memory_bank_mb": fit_stats["memory_bank_mb"],
        "fit_sec": fit_sec,
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)) if ckpt_path is not None else np.nan,
    }


# Load the saved autoencoder checkpoint for one category.
def load_ae_checkpoint(category):
    ckpt_path = AE_CHECKPOINTS_DIR / f"ae_{category}.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"Autoencoder checkpoint was not found at {ckpt_path}. "
            "Run notebook 03 if you want to evaluate autoencoder thresholds."
        )

    model = SmallAE()
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state, strict=True)
    model.eval()
    return model, ckpt_path


# Rebuild the saved autoencoder and return a score function in the same format.
def build_autoencoder_runtime(category):
    _, val_loader, test_loader = make_split_loaders(category, input_kind="ae")
    model, ckpt_path = load_ae_checkpoint(category)
    model = model.to(DEVICE)

    def score_fn(images):
        return ae_scores(model, images)

    return {
        "train_loader": None,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "model": model,
        "features": None,
        "handles": [],
        "score_fn": score_fn,
        "feature_dim": np.nan,
        "memory_bank_n": np.nan,
        "memory_bank_mb": np.nan,
        "fit_sec": np.nan,
        "peak_gpu_mem_mb": np.nan,
        "checkpoint_size_mb": float(file_size_mb(ckpt_path)),
    }


# Route each method name to the matching runtime builder.
def build_method_runtime(category, method_name):
    meta = parse_method_name(method_name)

    if method_name == "autoencoder":
        runtime = build_autoencoder_runtime(category)
    else:
        runtime = build_feature_method_runtime(
            category=category,
            representation_source=meta["representation_source"],
            anomaly_head=meta["anomaly_head"],
            aug_strength=meta["aug_strength"] if meta["representation_source"] == "simclr" else None,
        )

    return {**meta, **runtime}


# 6.0 Select the failure-analysis target

## Purpose
- identify the strongest overall method from the saved main comparison table
- identify the weakest category for that selected method
- load the saved balanced threshold for that exact method-category pair


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Select the method, category, and threshold for failure analysis
#------------------------------------------------------------------------------
rank_main = table_main_comparison_mean.dropna(subset=["img_pr_auc", "img_roc_auc"], how="all").copy()
if len(rank_main) == 0:
    raise RuntimeError(
        "The main comparison mean table does not contain valid ranking metrics. "
        "Rerun notebook 05 before continuing."
    )

rank_main = rank_main.sort_values(["img_pr_auc", "img_roc_auc", "px_roc_auc"], ascending=False).reset_index(drop=True)
BEST_OVERALL_METHOD = str(rank_main.iloc[0]["method"])

best_overall_by_cat = table_main_comparison_full[
    (table_main_comparison_full["method"] == BEST_OVERALL_METHOD) &
    (table_main_comparison_full["category"] != "MEAN")
].copy()

if len(best_overall_by_cat) == 0:
    raise RuntimeError(
        f"No category-level rows were found for {BEST_OVERALL_METHOD} in the main comparison full table."
    )

best_overall_by_cat = best_overall_by_cat.sort_values(["img_pr_auc", "img_roc_auc", "px_roc_auc"], ascending=[True, True, True]).reset_index(drop=True)
FAIL_CATEGORY = str(best_overall_by_cat.iloc[0]["category"])

threshold_match = table_thresholds_category[
    (table_thresholds_category["target_name"] == "best_overall") &
    (table_thresholds_category["method"] == BEST_OVERALL_METHOD) &
    (table_thresholds_category["category"] == FAIL_CATEGORY) &
    (table_thresholds_category["policy"] == FAILURE_POLICY)
].copy()

if len(threshold_match) == 0:
    raise RuntimeError(
        "The required balanced threshold was not found in notebook 06 outputs for "
        f"method={BEST_OVERALL_METHOD}, category={FAIL_CATEGORY}, policy={FAILURE_POLICY}."
    )

FAIL_THRESHOLD = float(threshold_match.iloc[0]["threshold"])

failure_selection = {
    "best_overall_method": BEST_OVERALL_METHOD,
    "failure_category": FAIL_CATEGORY,
    "policy": FAILURE_POLICY,
    "threshold": FAIL_THRESHOLD,
}
save_json_overwrite(failure_selection, FAILURE_SELECTION_PATH)

print_banner("Failure-analysis selection")
print("BEST_OVERALL_METHOD :", BEST_OVERALL_METHOD)
print("FAIL_CATEGORY       :", FAIL_CATEGORY)
print("FAILURE_POLICY      :", FAILURE_POLICY)
print("FAIL_THRESHOLD      :", FAIL_THRESHOLD)
print("Saved selection to  :", FAILURE_SELECTION_PATH)

display(best_overall_by_cat[[
    "method",
    "category",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "img_pr_auc",
    "img_roc_auc",
    "px_roc_auc",
]].head(10))


# 7.0 Run the failure analysis

## Purpose
- rebuild the selected runtime only once
- score the full test set for the chosen category
- assign predicted labels and error types using the saved threshold
- export detailed and summary tables for later use in the README or report


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Rebuild the runtime, collect detailed predictions, and save tables
#------------------------------------------------------------------------------
runtime = build_method_runtime(FAIL_CATEGORY, BEST_OVERALL_METHOD)
detailed_rows = collect_detailed_preds(runtime["test_loader"], runtime["score_fn"])

for row in detailed_rows:
    row["pred"] = int(row["score"] >= FAIL_THRESHOLD)
    row["error_type"] = (
        "FP" if (row["label"] == 0 and row["pred"] == 1) else
        "FN" if (row["label"] == 1 and row["pred"] == 0) else
        "TP" if (row["label"] == 1 and row["pred"] == 1) else
        "TN"
    )
    row["defect_type"] = Path(row["path"]).parent.name
    row["file_name"] = Path(row["path"]).name

df_failures = pd.DataFrame([
    {
        "method": BEST_OVERALL_METHOD,
        "category": FAIL_CATEGORY,
        "policy": FAILURE_POLICY,
        "threshold": FAIL_THRESHOLD,
        "path": row["path"],
        "file_name": row["file_name"],
        "defect_type": row["defect_type"],
        "label": row["label"],
        "pred": row["pred"],
        "score": row["score"],
        "error_type": row["error_type"],
    }
    for row in detailed_rows
])

if len(df_failures) == 0:
    raise RuntimeError("No failure-analysis rows were collected from the rebuilt runtime.")

df_failure_counts = (
    df_failures
    .groupby(["method", "category", "error_type"], as_index=False)
    .size()
    .rename(columns={"size": "n_images"})
    .sort_values(["error_type"])
    .reset_index(drop=True)
)

save_csv_overwrite(df_failures, TABLE_FAILURES_PATH)
save_csv_overwrite(df_failure_counts, TABLE_FAILURE_COUNTS_PATH)

if runtime.get("handles", None) is not None:
    remove_handles(runtime["handles"])

if "model" in runtime and runtime["model"] is not None:
    del runtime["model"]

if DEVICE == "cuda":
    torch.cuda.empty_cache()

print_banner("Saved failure-analysis tables")
print("Detailed failure table :", TABLE_FAILURES_PATH)
print("Failure counts table   :", TABLE_FAILURE_COUNTS_PATH)

display(df_failure_counts)
display(df_failures.head(10))


# 8.0 Save the qualitative failure figure

## Purpose
- create a compact figure with representative true positives, false positives, and false negatives
- make it easier to inspect how the selected method behaves on harder cases without opening the full notebook


In [ ]:
#------------------------------------------------------------------------------
# 8.1 Save the representative failure figure
#------------------------------------------------------------------------------
fps = sorted([row for row in detailed_rows if row["error_type"] == "FP"], key=lambda row: row["score"], reverse=True)[:TOP_FP_N]
fns = sorted([row for row in detailed_rows if row["error_type"] == "FN"], key=lambda row: row["score"])[:TOP_FN_N]
tps = sorted([row for row in detailed_rows if row["error_type"] == "TP"], key=lambda row: row["score"], reverse=True)[:TOP_TP_N]

examples = tps + fps + fns
if len(examples) == 0:
    anomalies = [row for row in detailed_rows if row["label"] == 1]
    examples = sorted(anomalies, key=lambda row: row["score"], reverse=True)[:4]

if len(examples) == 0:
    raise RuntimeError("No examples were available for the qualitative failure figure.")

n_examples = len(examples)
fig = plt.figure(figsize=(12, max(3.0, 3.0 * n_examples)))

for idx, row in enumerate(examples):
    caption = f"{row['error_type']} | {row['defect_type']} | score={row['score']:.3f}"

    ax1 = plt.subplot(n_examples, 3, idx * 3 + 1)
    overlay(ax1, row["img_tensor"], None, title="Image")
    ax1.set_ylabel(caption, fontsize=9)

    ax2 = plt.subplot(n_examples, 3, idx * 3 + 2)
    show_mask(ax2, row["mask"], title="GT mask")

    ax3 = plt.subplot(n_examples, 3, idx * 3 + 3)
    overlay(ax3, row["img_tensor"], row["heat"], title="Heatmap")

plt.tight_layout()
ensure_parent(FIG_FAILURES_PATH)
plt.savefig(FIG_FAILURES_PATH, dpi=220, bbox_inches="tight")
plt.show()

print_banner("Saved failure-analysis figure")
print("Failure figure path:", FIG_FAILURES_PATH)


# 9.0 Final review

## Purpose
- print the saved artefacts in one place
- surface the main failure tables at the end of the notebook for quick review


In [ ]:
#------------------------------------------------------------------------------
# 9.1 Final saved artefacts
#------------------------------------------------------------------------------
run_metadata = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": int(GPU_COUNT),
    "dataset_root": str(MVTEC_DIR),
    "best_overall_method": BEST_OVERALL_METHOD,
    "failure_category": FAIL_CATEGORY,
    "failure_policy": FAILURE_POLICY,
    "failure_threshold": FAIL_THRESHOLD,
    "n_failure_rows": int(len(df_failures)),
    "n_failure_count_rows": int(len(df_failure_counts)),
}
save_json_overwrite(run_metadata, RUN_METADATA_PATH)

print_banner("Saved artefacts")
print("Run metadata path       :", RUN_METADATA_PATH)
print("Failure selection path  :", FAILURE_SELECTION_PATH)
print("Detailed failure table  :", TABLE_FAILURES_PATH)
print("Failure counts table    :", TABLE_FAILURE_COUNTS_PATH)
print("Failure figure path     :", FIG_FAILURES_PATH)

display(df_failure_counts)
display(df_failures.head(10))
